In [6]:
import cv2
from ultralytics import YOLO
import supervision as sv
import numpy as np
import math
import time

camera = cv2.VideoCapture(0)
model = YOLO("yolov8x.pt")

bounding_box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_position=sv.Position.TOP_LEFT)
percentage_bar_annotator = sv.BoxAnnotator()

while True:
    success, frame = camera.read() 
    result = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    coord_center = []
    for i in range (len(detections)):
        centers = (detections.xyxy[:,:2][i]+detections.xyxy[:,2:][i])/2
        coord_center.append(centers)
    coord_center=np.array(coord_center)  
    if coord_center.size != 0:
        print(coord_center)
        cv2.circle(frame, (int(coord_center[0][0]),int(coord_center[0][1])), 3, (118, 103, 154), 3)
    frame = bounding_box_annotator.annotate(
        scene=frame,
        detections=detections
    )

    frame = label_annotator.annotate(
        scene=frame,
        detections=detections
    )
    frame = percentage_bar_annotator.annotate(
        scene=frame,
        detections=detections
    )
    cv2.imshow("yolov8", frame)

    if (cv2.waitKey(30) == 27):
        break

[[     316.97      226.35]]
[[     317.42      226.88]]
[[      317.2      227.41]]
[[     316.91      226.86]]
[[     316.91      226.57]]
[[     316.85      227.49]]
[[      316.4      227.38]]
[[     316.85         227]]
[[     316.82      226.87]]
[[     317.22      226.98]]
[[     316.82      227.53]]
[[     316.63      226.67]]
[[     316.88      227.94]]
[[     316.73      227.15]]
[[     317.11         227]]
[[     316.91       227.3]]
[[      316.8      227.38]]


KeyboardInterrupt: 

In [ ]:
import cv2
from realsense_camera import *
from ultralytics import YOLO
import supervision as sv
import numpy as np
import math
import pyrealsense2
from kuka import Kuka
import time
from openshowvar import *


#Connect robot
robot = openshowvar(ip = '192.168.17.2', port = 7000)
kuka = Kuka(robot)

kuka.set_base(8)
kuka.set_tool(10)

# Set speed (%)
kuka.set_speed(100)

kuka.read_cartesian()
x0=282.73
y0=122.48
z0=451.23
# Построение траектории
trajectory_arr = []
# Построение квадрата
trajectory_arr.append(np.array([kuka.x_cartesian, kuka.y_cartesian, kuka.z_cartesian, kuka.A_cartesian, kuka.B_cartesian, kuka.C_cartesian]))
trajectory_arr.append(np.array([x0, y0, z0, 90.57, 0, 180]))
trajectory_arr = np.array(trajectory_arr)
# Отправка траектории роботу
kuka.lin_continuous_massiv(trajectory_arr)

fx=604.602
fy=604.162
Cx=320
Cy=240
cap = RealsenseCamera()
#camera = cv2.VideoCapture(0)
model = YOLO("yolov8x.pt")

bounding_box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_position=sv.Position.TOP_LEFT)
percentage_bar_annotator = sv.BoxAnnotator()
print('start')
while True:
    success, depth_colormap, frame, depth = cap.get_frame_stream()
    #success, frame = camera.read() 
    result = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    #classes = result[0].boxes.cls.tolist() #Класс
    #print(classes)
    coord_center = []
    for i in range (len(detections)):
        centers = (detections.xyxy[:,:2][i]+detections.xyxy[:,2:][i])/2
        coord_center.append(centers)
    coord_center=np.array(coord_center)  
    if coord_center.size != 0:
        print(coord_center)
        cv2.circle(frame, (int(coord_center[0][0]),int(coord_center[0][1])), 3, (118, 103, 154), 3)
        for i in range(len(detections)):
            X_mm=(depth[int(coord_center[0][1]), int(coord_center[0][0])])*((Cx-coord_center[0][0])/fx)
            Y_mm=(depth[int(coord_center[0][1]), int(coord_center[0][0])])*((Cy-coord_center[0][1])/fy)
            Y_mm=Y_mm-62
            Z_mm=depth[int(coord_center[0][1]), int(coord_center[0][0])]-90
            print('Xmm',X_mm)
            print('Ymm',Y_mm)
            print('Zmm',Z_mm)
    frame = bounding_box_annotator.annotate(
        scene=frame,
        detections=detections
    )

    frame = label_annotator.annotate(
        scene=frame,
        detections=detections
    )
    frame = percentage_bar_annotator.annotate(
        scene=frame,
        detections=detections
    )
    cv2.imshow("yolov8", frame)

    if (cv2.waitKey(30) == 27):
        break

start
[[      337.8      233.41]]
Xmm -14.511843
Ymm -56.620094
Zmm 51
[[     338.12      232.67]]
Xmm -14.777833
Ymm -56.02106
Zmm 51
[[     338.17      233.64]]
Xmm -14.81633
Ymm -56.810947
Zmm 51
[[     337.72      233.17]]
Xmm -14.445252
Ymm -56.430473
Zmm 51
[[     337.71      233.14]]
Xmm -14.43948
Ymm -56.399296
Zmm 51
[[     337.92      234.03]]
Xmm -14.6149645
Ymm -57.128765
Zmm 51
[[     337.74      233.71]]
Xmm -14.468644
Ymm -56.8647
Zmm 51
[[     337.95      233.66]]
Xmm -14.637809
Ymm -56.828613
Zmm 51
[[      337.8      233.47]]
Xmm -14.516671
Ymm -56.668415
Zmm 51
[[     338.09      233.57]]
Xmm -14.747897
Ymm -56.75092
Zmm 51
[[        338      233.44]]
Xmm -14.676503
Ymm -56.644623
Zmm 51
[[     337.65      233.39]]
Xmm -14.393244
Ymm -56.603794
Zmm 51
[[     337.73      232.28]]
Xmm -14.461129
Ymm -55.696495
Zmm 51
[[     337.51      233.32]]
Xmm -14.277083
Ymm -56.548637
Zmm 51
[[     337.51      232.94]]
Xmm -14.273898
Ymm -56.238823
Zmm 51
[[      337.8      233.5

KeyboardInterrupt: 

In [26]:
import cv2
from realsense_camera import *
from ultralytics import YOLO
import supervision as sv
import numpy as np
import math
import pyrealsense2
from kuka import Kuka
import time
from openshowvar import *


#Connect robot
robot = openshowvar(ip = '192.168.17.2', port = 7000)
kuka = Kuka(robot)

kuka.set_base(8)
kuka.set_tool(10)

# Set speed (%)
kuka.set_speed(50)

kuka.read_cartesian()
x0=282.73
y0=122.48
z0=451.23
# Чтение текущих координат робота
kuka.read_cartesian()
# Построение траектории
first_point=[x0, y0, z0, 90.57, 0, 180]
trajectory=np.array([first_point])
kuka.lin_continuous(kuka,trajectory)

fx=604.602
fy=604.162
Cx=320
Cy=240
cap = RealsenseCamera()
#camera = cv2.VideoCapture(0)
model = YOLO("yolov8x.pt")

bounding_box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_position=sv.Position.TOP_LEFT)
percentage_bar_annotator = sv.BoxAnnotator()
print('start')
while True:
    success, depth_colormap, frame, depth = cap.get_frame_stream()
    #success, frame = camera.read() 
    result = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    #classes = result[0].boxes.cls.tolist() #Класс
    #print(classes)
    coord_center = []
    for i in range (len(detections)):
        centers = (detections.xyxy[:,:2][i]+detections.xyxy[:,2:][i])/2
        coord_center.append(centers)
    coord_center=np.array(coord_center)  
    if coord_center.size != 0:
        print(coord_center)
        cv2.circle(frame, (int(coord_center[0][0]),int(coord_center[0][1])), 3, (118, 103, 154), 3)
        for i in range(len(detections)):
            X_mm=(depth[int(coord_center[0][1]), int(coord_center[0][0])])*((Cx-coord_center[0][0])/fx)
            Y_mm=(depth[int(coord_center[0][1]), int(coord_center[0][0])])*((Cy-coord_center[0][1])/fy)
            Y_mm=Y_mm+62
            Z_mm=depth[int(coord_center[0][1]), int(coord_center[0][0])]-50
            print('Xmm',X_mm)
            print('Ymm',Y_mm)
            print('Zmm',Z_mm)
            
            # Движение Kuka
            kuka.open_grip()
            # Чтение текущих координат робота
            kuka.read_cartesian()
            # Построение траектории
            first_point=[kuka.x_cartesian-X_mm, kuka.y_cartesian+Y_mm, kuka.z_cartesian-Z_mm,kuka.A_cartesian, 0, kuka.C_cartesian]
            trajectory=np.array([first_point])
            kuka.lin_continuous(kuka,trajectory)
            kuka.close_grip()
            time.sleep(1)
            # Чтение текущих координат робота
            kuka.read_cartesian()
            # Построение траектории
            first_point=[x0, y0, z0, 90.57, 0, 180]
            trajectory=np.array([first_point])
            kuka.lin_continuous(kuka,trajectory)
            
    frame = bounding_box_annotator.annotate(
        scene=frame,
        detections=detections
    )

    frame = label_annotator.annotate(
        scene=frame,
        detections=detections
    )
    frame = percentage_bar_annotator.annotate(
        scene=frame,
        detections=detections
    )
    cv2.imshow("yolov8", frame)

    if (cv2.waitKey(30) == 27):
        break

start
[[     417.07      204.37]]
Xmm -78.02649
Ymm 90.66066
Zmm 436


KeyboardInterrupt: 

In [30]:
import cv2
from realsense_camera import *
from ultralytics import YOLO
import supervision as sv
import numpy as np
import math
import pyrealsense2
from kuka import Kuka
import time
from openshowvar import *


#Connect robot
robot = openshowvar(ip = '192.168.17.2', port = 7000)
kuka = Kuka(robot)

kuka.set_base(8)
kuka.set_tool(10)

# Set speed (%)
kuka.set_speed(50)

kuka.read_cartesian()
x0=282.73
y0=122.48
z0=451.23
# Чтение текущих координат робота
kuka.read_cartesian()
# Построение траектории
first_point=[x0, y0, z0, 90.57, 0, 180]
trajectory=np.array([first_point])
kuka.lin_continuous(kuka,trajectory)

fx=604.602
fy=604.162
Cx=320
Cy=240
cap = RealsenseCamera()
#camera = cv2.VideoCapture(0)
model = YOLO("yolov8x-seg.pt")

def getOrientation(pts, img):
    rect = cv2.minAreaRect(pts)
    width = int(rect[1][0])
    height = int(rect[1][1])
    angle = int(rect[2])
    if width < height:
        angle = 90- angle
    else:
        angle = 90+(90-angle)
    return angle,width, height
print('start')
while True:
    success, depth_colormap, frame, depth = cap.get_frame_stream()
    #success, frame = camera.read() 
    results = model(frame, verbose=False)
    boxes = results[0].boxes.xyxy.tolist() #Рамка
    #print(boxes)
    if boxes!=[]:
        frame=results[0].plot()
        classes = results[0].boxes.cls.tolist() #Класс
        #print(classes)
        confidences = results[0].boxes.conf.tolist()
        #print(confidences)
        mask_imshow = results[0].masks[0].data.cpu().numpy().transpose(1, 2, 0)#Маска двухбитная 
        mask_imshow = results[0].masks[0].data.cpu().numpy().transpose(1, 2, 0)#Маска двухбитная 

        # 1. Берём маску и убираем лишнюю ось
        mask_imshow = results[0].masks.data[0].cpu().numpy()  # shape: (H_mask, W_mask)
        
        # 2. Ресайзим под размер кадра
        mask_resized = cv2.resize(mask_imshow, (frame.shape[1], frame.shape[0]),
                                interpolation=cv2.INTER_NEAREST)
        
        # 3. Нормализуем в 0..255 и делаем uint8
        mask_uint8 = (mask_resized * 255).astype(np.uint8)
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL,
                               cv2.CHAIN_APPROX_NONE) 
        if len(contours) != 0: 
            maxc = max(contours, key=cv2.contourArea)
            cv2.drawContours(frame, maxc, -1, (0, 0, 255), 2)
        rotation,width,height=getOrientation(maxc, frame)
        print(rotation)
        coord_center = []
        for i in range (len(boxes)):
            x = (boxes[i][0]+boxes[i][2])/2
            y= (boxes[i][1]+boxes[i][3])/2
            coord_center.append([x,y])
        coord_center=np.array(coord_center)  
        if coord_center.size != 0:
            print(coord_center)
            cv2.circle(frame, (int(coord_center[0][0]),int(coord_center[0][1])), 3, (118, 103, 154), 3)
            for i in range(len(boxes)):
                X_mm=(depth[int(coord_center[0][1]), int(coord_center[0][0])])*((Cx-coord_center[0][0])/fx)
                Y_mm=(depth[int(coord_center[0][1]), int(coord_center[0][0])])*((Cy-coord_center[0][1])/fy)
                Y_mm=Y_mm+62
                Z_mm=depth[int(coord_center[0][1]), int(coord_center[0][0])]-50
                print('Xmm',X_mm)
                print('Ymm',Y_mm)
                print('Zmm',Z_mm)
                
                # Движение Kuka
                kuka.open_grip()
                # Чтение текущих координат робота
                kuka.read_cartesian()
                # Построение траектории
                first_point=[kuka.x_cartesian-X_mm, kuka.y_cartesian+Y_mm, kuka.z_cartesian-Z_mm+100,kuka.A_cartesian, 0, kuka.C_cartesian]
                second_point=[kuka.x_cartesian-X_mm, kuka.y_cartesian+Y_mm, kuka.z_cartesian-Z_mm+100,rotation, 0, kuka.C_cartesian]
                third_point=[kuka.x_cartesian-X_mm, kuka.y_cartesian+Y_mm, kuka.z_cartesian-Z_mm,rotation, 0, kuka.C_cartesian]
                trajectory=np.array([first_point, second_point, third_point])
                kuka.lin_continuous(kuka,trajectory)
                kuka.close_grip()
                time.sleep(1)
                # Чтение текущих координат робота
                kuka.read_cartesian()
                # Построение траектории
                first_point=[x0, y0, z0, 90.57, 0, 180]
                trajectory=np.array([first_point])
                kuka.lin_continuous(kuka,trajectory)
                kuka.read_cartesian()
                # Построение траектории
                first_point=[kuka.x_cartesian-X_mm, kuka.y_cartesian+Y_mm, kuka.z_cartesian-Z_mm+100,rotation, 0, kuka.C_cartesian]
                second_point=[kuka.x_cartesian-X_mm, kuka.y_cartesian+Y_mm, kuka.z_cartesian-Z_mm,rotation, 0, kuka.C_cartesian]
                trajectory=np.array([first_point, second_point])
                kuka.lin_continuous(kuka,trajectory)
                kuka.open_grip()
                time.sleep(1)
                kuka.read_cartesian()
                # Построение траектории
                first_point=[x0, y0, z0, 90.57, 0, 180]
                trajectory=np.array([first_point])
                kuka.lin_continuous(kuka,trajectory)
            
        cv2.imshow("mask_imshow", mask_uint8)
    cv2.imshow("yolov8", frame)

    if (cv2.waitKey(30) == 27):
        break

start
242
[[      367.2      197.67]]
Xmm -38.334613347170176
Ymm 96.40037864935523
Zmm 441


KeyboardInterrupt: 